# Networks for MAPPO — Actor & Critic

Defines the neural networks used in the MAPPO algorithm for the  
LLM-Empowered Multi-UAV Active Sensing prototype.

### Architecture
| Network | Input | Hidden | Output |
|---------|-------|--------|---------|
| Actor (per UAV) | 17 | 64 → 32 | 5 (action probs) |
| Critic (centralized) | 34 (joint) | 64 → 64 | 1 (state value) |

### Observation vector per UAV (17 values)
```
[row_norm, col_norm,          (2)  UAV position
 energy_norm,                (1)  battery level
 w[0]...w[11],              (12)  risk weights for all sectors
 rel_row, rel_col]           (2)  relative position of other UAV
```

### Actions (discrete, 2D flat plane)
```
0 = stay
1 = north  (row - 1)
2 = south  (row + 1)
3 = west   (col - 1)
4 = east   (col + 1)
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device    : {DEVICE}")

## Constants

In [ ]:
OBS_SIZE    = 17   # observation size per UAV
JOINT_SIZE  = 34   # OBS_SIZE x 2 UAVs (critic input)
N_ACTIONS   = 5    # stay, north, south, west, east

## Actor Network

Each UAV has its own local Actor.  
Maps local observation → action probability distribution.

```
Linear(17 → 64) → ReLU
Linear(64 → 32) → ReLU
Linear(32 →  5) → Softmax
```

During **training**: actions are sampled stochastically from the distribution.  
During **evaluation**: argmax (most probable action) is selected.

In [ ]:
class ActorNetwork(nn.Module):
    """
    Local actor for one UAV.
    Input  : local observation (17 values)
    Output : probability distribution over 5 discrete actions
    """

    def __init__(self, obs_size=OBS_SIZE, n_actions=N_ACTIONS):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, n_actions)
        )

    def forward(self, obs):
        """
        Args:
            obs : Tensor of shape (batch, 17) or (17,)
        Returns:
            Categorical distribution over actions
        """
        logits = self.net(obs)
        return Categorical(logits=logits)

    def get_action(self, obs):
        """
        Sample an action and return it with its log probability.
        Used during rollout collection (training).

        Returns:
            action   : int
            log_prob : Tensor scalar
        """
        dist     = self.forward(obs)
        action   = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob

    def get_log_prob_entropy(self, obs, actions):
        """
        Returns log probabilities and entropy for given (obs, action) pairs.
        Used during PPO update step.

        Args:
            obs     : Tensor (batch, 17)
            actions : Tensor (batch,)
        Returns:
            log_probs : Tensor (batch,)
            entropy   : Tensor scalar  (used for optional entropy bonus)
        """
        dist      = self.forward(obs)
        log_probs = dist.log_prob(actions)
        entropy   = dist.entropy().mean()
        return log_probs, entropy

    def greedy_action(self, obs):
        """
        Returns the most probable action (no sampling).
        Used during evaluation.
        """
        with torch.no_grad():
            dist = self.forward(obs)
            return torch.argmax(dist.probs).item()

## Critic Network

Single centralized Critic shared by both UAVs.  
Takes the **joint state** (both UAV observations concatenated) → scalar value.

```
Linear(34 → 64) → ReLU
Linear(64 → 64) → ReLU
Linear(64 →  1)
```

Only used during **training** (CTDE — Centralized Training, Decentralized Execution).  
Discarded after training; UAVs run using only their local Actor.

In [ ]:
class CriticNetwork(nn.Module):
    """
    Centralized critic for MAPPO.
    Input  : joint observation of all UAVs (34 values = 17 x 2)
    Output : scalar state value V(S_t)
    """

    def __init__(self, joint_size=JOINT_SIZE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(joint_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, joint_obs):
        """
        Args:
            joint_obs : Tensor (batch, 34) or (34,)
        Returns:
            value     : Tensor (batch, 1)
        """
        return self.net(joint_obs)

## Sanity Check
Verify both networks produce the correct output shapes.

In [ ]:
import numpy as np

actor  = ActorNetwork().to(DEVICE)
critic = CriticNetwork().to(DEVICE)

# Dummy inputs
dummy_obs       = torch.randn(1, OBS_SIZE).to(DEVICE)
dummy_joint_obs = torch.randn(1, JOINT_SIZE).to(DEVICE)

# Actor forward
dist   = actor(dummy_obs)
action, log_prob = actor.get_action(dummy_obs.squeeze())

# Critic forward
value  = critic(dummy_joint_obs)

print("=" * 40)
print("Actor")
print(f"  Input  shape : {dummy_obs.shape}")
print(f"  Action probs : {dist.probs.detach().cpu().numpy().round(3)}")
print(f"  Sampled action : {action}")
print(f"  Log prob : {log_prob.item():.4f}")
print()
print("Critic")
print(f"  Input  shape : {dummy_joint_obs.shape}")
print(f"  Value output : {value.item():.4f}")
print("=" * 40)

## Parameter Count

In [ ]:
def count_params(model, name):
    total = sum(p.numel() for p in model.parameters())
    print(f"{name:<20} : {total:,} parameters")

count_params(actor,  "ActorNetwork")
count_params(critic, "CriticNetwork")

# Two actors (one per UAV) + one critic
total = 2 * sum(p.numel() for p in actor.parameters()) + \
            sum(p.numel() for p in critic.parameters())
print(f"{'Total (2 actors + critic)':<20} : {total:,} parameters")

## Save Network Classes
Export to a `.py` file so `02_mappo.ipynb` can import them directly.

In [ ]:
network_code = '''
import torch
import torch.nn as nn
from torch.distributions import Categorical

OBS_SIZE   = 17
JOINT_SIZE = 34
N_ACTIONS  = 5

class ActorNetwork(nn.Module):
    def __init__(self, obs_size=OBS_SIZE, n_actions=N_ACTIONS):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, 64), nn.ReLU(),
            nn.Linear(64, 32),       nn.ReLU(),
            nn.Linear(32, n_actions)
        )

    def forward(self, obs):
        return Categorical(logits=self.net(obs))

    def get_action(self, obs):
        dist = self.forward(obs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)

    def get_log_prob_entropy(self, obs, actions):
        dist = self.forward(obs)
        return dist.log_prob(actions), dist.entropy().mean()

    def greedy_action(self, obs):
        with torch.no_grad():
            return torch.argmax(self.forward(obs).probs).item()


class CriticNetwork(nn.Module):
    def __init__(self, joint_size=JOINT_SIZE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(joint_size, 64), nn.ReLU(),
            nn.Linear(64, 64),         nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, joint_obs):
        return self.net(joint_obs)
'''

with open('networks.py', 'w') as f:
    f.write(network_code.strip())

print('networks.py saved — ready to import in 02_mappo.ipynb')